In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc imbalanced-learn matplotlib numpy pandas qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 하이브리드 양자 강화 앙상블 분류 (그리드 안정성 워크플로)

*사용량 예상: Eagle r3 프로세서에서 각 작업당 QPU 시간 20분. (참고: 이는 예상치일 뿐이며, 실제 실행 시간은 다를 수 있습니다.)*
## 배경
이 튜토리얼은 고전 앙상블을 양자 최적화 단계로 강화하는 하이브리드 양자-고전 워크플로를 시연합니다. Multiverse Computing의 "Singularity Machine Learning – Classification" (Qiskit 함수)을 사용하여, 기존 학습기 풀(예: 결정 트리, k-NN, 로지스틱 회귀)을 훈련한 다음 양자 레이어로 해당 풀을 정제하여 다양성과 일반화를 개선합니다. 목표는 실용적입니다: 실제 그리드 안정성 예측 작업에서, 동일한 데이터 분할 하에 강력한 고전 기준선과 양자 최적화 대안을 비교하여 양자 단계가 어디에 도움이 되는지, 그리고 그 비용이 무엇인지 확인할 수 있습니다.

이것이 중요한 이유: 많은 약한 학습기 중에서 좋은 부분 집합을 선택하는 것은 앙상블 크기가 커질수록 빠르게 증가하는 조합 문제입니다. 부스팅, 배깅, 스태킹과 같은 고전적 휴리스틱은 중간 규모에서는 잘 작동하지만, 대규모의 중복된 모델 라이브러리를 효율적으로 탐색하는 데는 어려움을 겪을 수 있습니다. 이 함수는 양자 알고리즘(특히 QAOA, 다른 구성에서는 선택적으로 VQE)을 통합하여 고전 학습기가 훈련된 후 해당 공간을 더 효과적으로 탐색하고, 더 잘 일반화되는 소형의 다양한 부분 집합을 찾을 가능성을 높입니다.

중요하게도, 데이터 규모는 Qubit에 의해 제한되지 않습니다. 데이터에 대한 주요 작업인 전처리, 학습기 풀 훈련, 평가는 고전적으로 유지되며 수백만 개의 예시를 처리할 수 있습니다. Qubit는 양자 선택 단계에서 사용되는 앙상블 크기만 결정합니다. 이러한 분리가 오늘날의 하드웨어에서 이 접근 방식을 실용적으로 만드는 요소입니다: 데이터 및 모델 훈련을 위해 익숙한 scikit-learn 워크플로를 유지하면서 Qiskit Functions의 깔끔한 액션 인터페이스를 통해 양자 단계를 호출할 수 있습니다.

실제로, 앙상블에 다양한 학습기 유형(예: 결정 트리, 로지스틱 회귀, k-NN)을 제공할 수 있지만, 결정 트리가 가장 좋은 성능을 보이는 경향이 있습니다. 최적화기는 일관되게 더 강한 앙상블 구성원을 선호하며, 이질적인 학습기가 제공되면 선형 회귀와 같은 약한 모델은 일반적으로 결정 트리와 같이 더 표현력 있는 모델에 의해 제거됩니다.

여기서 수행할 작업: 그리드 안정성 데이터셋을 준비하고 균형을 맞춥니다; 고전 AdaBoost 기준선을 수립합니다; 앙상블 너비와 정규화를 변화시키는 여러 양자 구성을 실행합니다; Qiskit Serverless를 통해 IBM&reg; 시뮬레이터 또는 QPU에서 실행합니다; 그리고 모든 실행에 걸쳐 정확도, 정밀도, 재현율, F1을 비교합니다. 과정에서 함수의 액션 패턴(`create`, `fit`, `predict`, `fit_predict`, `create_fit_predict`)과 주요 제어 항목을 사용합니다:
- 정규화 유형: 직접 희소성을 위한 `onsite` (λ) 및 상호작용 항과 onsite 항 간의 비율 기반 트레이드오프를 위한 `alpha`
- 자동 정규화: 희소성을 자동으로 조정하기 위해 목표 선택 비율과 함께 `regularization="auto"` 설정
- 최적화기 옵션: 시뮬레이터 대 QPU, 반복 횟수, 고전 최적화기 및 해당 옵션, 트랜스파일 깊이, 런타임 Sampler/Estimator 설정

문서의 벤치마크에 따르면 어려운 문제에서 학습기(Qubit) 수가 증가할수록 정확도가 향상되며, 양자 분류기는 비교 가능한 고전 앙상블과 동등하거나 이를 능가합니다. 이 튜토리얼에서는 워크플로를 처음부터 끝까지 재현하고, 앙상블 너비를 늘리거나 적응형 정규화로 전환하는 것이 합리적인 자원 사용으로 더 나은 F1을 제공하는 시점을 살펴봅니다. 결과는 양자 최적화 단계가 실제 응용 프로그램에서 고전 앙상블 학습을 대체하는 것이 아니라 보완할 수 있는 방법에 대한 기반 있는 관점을 제공합니다.
## 요구 사항
이 튜토리얼을 시작하기 전에 Python 환경에 다음 패키지가 설치되어 있는지 확인하세요:

- `qiskit[visualization]~=2.1.0`
- `qiskit-serverless~=0.24.0`
- `qiskit-ibm-runtime v0.40.1`
- `qiskit-ibm-catalog~=0.8.0`
- `scikit-learn==1.5.2`
- `pandas>=2.0.0,<3.0.0`
- `imbalanced-learn~=0.12.3`
## 설정
이 섹션에서는 Qiskit Serverless 클라이언트를 초기화하고 Multiverse Computing에서 제공하는 Singularity Machine Learning – Classification 함수를 로드합니다.
Qiskit Serverless를 사용하면 리소스 관리에 대한 걱정 없이 IBM 관리형 클라우드 인프라에서 하이브리드 양자-고전 워크플로를 실행할 수 있습니다.
인증하고 Qiskit Functions에 액세스하려면 IBM Quantum Platform API 키와 클라우드 리소스 이름(CRN)이 필요합니다.
### 데이터셋 다운로드
이 튜토리얼을 실행하려면 레이블이 지정된 전력 시스템 센서 판독값이 포함된 전처리된 **그리드 안정성 분류 데이터셋**을 사용합니다.
다음 셀은 필요한 폴더 구조를 자동으로 생성하고 `wget`을 사용하여 훈련 및 테스트 파일을 환경에 직접 다운로드합니다.
이 파일이 이미 로컬에 있는 경우, 이 단계는 버전 일관성을 보장하기 위해 안전하게 덮어씁니다.

In [1]:
## Download dataset for Grid Stability Classification

# Create data directory if it doesn't exist
!mkdir -p data_tutorial/grid_stability

# Download the training and test sets from the official Qiskit documentation repo
!wget -q --show-progress -O data_tutorial/grid_stability/train.csv \
  https://raw.githubusercontent.com/Qiskit/documentation/main/datasets/tutorials/grid_stability/train.csv

!wget -q --show-progress -O data_tutorial/grid_stability/test.csv \
  https://raw.githubusercontent.com/Qiskit/documentation/main/datasets/tutorials/grid_stability/test.csv

# Check the files have been downloaded
!echo "Dataset files downloaded:"
!ls -lh data_tutorial/grid_stability/*.csv

data_tutorial/grid_ 100%[===================>] 612.94K  --.-KB/s    in 0.01s   
data_tutorial/grid_ 100%[===================>] 108.19K  --.-KB/s    in 0.006s  
Dataset files downloaded:
-rw-r--r-- 1 coder coder 109K Nov  8 18:50 data_tutorial/grid_stability/test.csv
-rw-r--r-- 1 coder coder 613K Nov  8 18:50 data_tutorial/grid_stability/train.csv


### Import required packages

In this section, we import all Python packages and Qiskit modules used throughout the tutorial.
These include core scientific libraries for data handling and model evaluation - such as `NumPy`, `pandas`, and `scikit-learn` - along with visualization tools and Qiskit components for running the quantum-enhanced model.
We also import the `QiskitRuntimeService` and `QiskitFunctionsCatalog` to connect with IBM Quantum&reg; services and access the Singularity Machine Learning function.

In [2]:
from typing import Tuple
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from qiskit_ibm_catalog import QiskitFunctionsCatalog
from qiskit_ibm_runtime import QiskitRuntimeService
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

### 필요한 패키지 가져오기
이 섹션에서는 튜토리얼 전반에 걸쳐 사용되는 모든 Python 패키지와 Qiskit 모듈을 가져옵니다.
여기에는 데이터 처리 및 모델 평가를 위한 핵심 과학 라이브러리(`NumPy`, `pandas`, `scikit-learn` 등)와 시각화 도구, 그리고 양자 강화 모델을 실행하기 위한 Qiskit 컴포넌트가 포함됩니다.
또한 IBM Quantum&reg; 서비스에 연결하고 Singularity Machine Learning 함수에 액세스하기 위해 `QiskitRuntimeService`와 `QiskitFunctionsCatalog`를 가져옵니다.

In [3]:
IBM_TOKEN = ""
IBM_INSTANCE_TEST = ""
IBM_INSTANCE_QUANTUM = ""
FUNCTION_NAME = "multiverse/singularity"
RANDOM_STATE: int = 123
TRAIN_PATH = "data_tutorial/grid_stability/train.csv"
TEST_PATH = "data_tutorial/grid_stability/test.csv"

### 상수 변수 설정

In [4]:
service = QiskitRuntimeService(
    token=IBM_TOKEN,
    channel="ibm_quantum_platform",
    instance=IBM_INSTANCE_QUANTUM,
)

backend = service.least_busy()
catalog = QiskitFunctionsCatalog(
    token=IBM_TOKEN,
    instance=IBM_INSTANCE_TEST,
    channel="ibm_quantum_platform",
)
singularity = catalog.load(FUNCTION_NAME)
print(
    "Successfully connected to IBM Qiskit Serverless and loaded the Singularity function."
)
print("Catalog:", catalog)
print("Singularity function:", singularity)

Successfully connected to IBM Qiskit Serverless and loaded the Singularity function.
Catalog: <QiskitFunctionsCatalog>
Singularity function: QiskitFunction(multiverse/singularity)


### IBM Quantum에 연결하고 Singularity 함수 로드
다음으로 IBM Quantum 서비스를 인증하고 Qiskit Functions Catalog에서 Singularity Machine Learning – Classification 함수를 로드합니다.
`QiskitRuntimeService`는 API 토큰과 인스턴스 CRN을 사용하여 IBM Quantum Platform에 보안 연결을 설정하여 양자 Backend에 대한 액세스를 허용합니다.
그런 다음 `QiskitFunctionsCatalog`를 사용하여 이름(`"multiverse/singularity"`)으로 Singularity 함수를 검색하여 나중에 하이브리드 양자-고전 계산을 위해 호출할 수 있습니다.
설정이 성공하면 함수가 올바르게 로드되었음을 나타내는 확인 메시지가 표시됩니다.

In [5]:
def load_data(data_path: str) -> Tuple[np.ndarray, np.ndarray]:
    """Load data from the given path to X and y arrays."""
    df: pd.DataFrame = pd.read_csv(data_path)
    return df.iloc[:, :-1].values, df.iloc[:, -1].values


def evaluate_predictions(predictions, y_true):
    """Compute and print accuracy, precision, recall, and F1 score."""
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions)
    recall = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    return accuracy, precision, recall, f1

## Step 1: Map classical inputs to a quantum problem

We begin by preparing the dataset for hybrid quantum–classical experimentation. The goal of this step is to convert the raw grid-stability data into balanced training, validation, and test splits that can be used consistently by both classical and quantum workflows. Maintaining identical splits ensures that later performance comparisons are fair and reproducible.

### Data loading and preprocessing

We first load the training and test CSV files, create a validation split, and balance the dataset using random over-sampling. Balancing prevents bias toward the majority class and provides a more stable learning signal for both classical and quantum ensemble models.

In [6]:
# Load and upload the data
X_train, y_train = load_data(TRAIN_PATH)
X_test, y_test = load_data(TEST_PATH)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE
)

# Balance the dataset through over-sampling of the positive class
ros = RandomOverSampler(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

print("Shapes:")
print("  X_train_bal:", X_train_bal.shape)
print("  y_train_bal:", y_train_bal.shape)
print("  X_val:", X_val.shape)
print("  y_val:", y_val.shape)
print("  X_test:", X_test.shape)
print("  y_test:", y_test.shape)

Shapes:
  X_train_bal: (5104, 12)
  y_train_bal: (5104,)
  X_val: (850, 12)
  y_val: (850,)
  X_test: (750, 12)
  y_test: (750,)


### 헬퍼 함수 정의
주요 실험을 실행하기 전에, 데이터 로딩과 모델 평가를 간소화하는 몇 가지 소규모 유틸리티 함수를 정의합니다.
- `load_data()`는 입력 CSV 파일을 NumPy 배열로 읽어 `scikit-learn` 및 양자 워크플로와의 호환성을 위해 특성과 레이블을 분리합니다.
- `evaluate_predictions()`는 주요 성능 지표인 정확도, 정밀도, 재현율, F1-점수를 계산하고 타이밍 정보가 제공되면 선택적으로 런타임을 보고합니다.

이러한 헬퍼 함수는 나중에 노트북에서 반복되는 작업을 단순화하고 고전 및 양자 분류기 모두에서 일관된 지표 보고를 보장합니다.

In [7]:
# ----- Classical baseline: AdaBoost -----
baseline = AdaBoostClassifier(n_estimators=60, random_state=RANDOM_STATE)
baseline.fit(X_train_bal, y_train_bal)
baseline_pred = baseline.predict(X_test)
print("Classical AdaBoost baseline:")
_ = evaluate_predictions(baseline_pred, y_test)

Classical AdaBoost baseline:
Accuracy: 0.7893333333333333
Precision: 1.0
Recall: 0.7893333333333333
F1: 0.8822652757078987


## Step 1: 고전 입력을 양자 문제로 매핑
하이브리드 양자-고전 실험을 위해 데이터셋을 준비하는 것으로 시작합니다. 이 단계의 목표는 원시 그리드 안정성 데이터를 고전 및 양자 워크플로 모두에서 일관되게 사용할 수 있는 균형 잡힌 훈련, 검증, 테스트 분할로 변환하는 것입니다. 동일한 분할을 유지하면 나중의 성능 비교가 공정하고 재현 가능합니다.
### 데이터 로딩 및 전처리
먼저 훈련 및 테스트 CSV 파일을 로드하고, 검증 분할을 생성하고, 무작위 오버 샘플링을 사용하여 데이터셋을 균형 있게 맞춥니다. 균형을 맞추면 다수 클래스에 대한 편향을 방지하고 고전 및 양자 앙상블 모델 모두에 더 안정적인 학습 신호를 제공합니다.

In [8]:
# QAOA / runtime configuration for best results on hardware
optimizer_options = {
    "simulator": False,  # set True to test locally without QPU
    "num_solutions": 100_000,  # broaden search over candidate ensembles
    "reps": 3,  # QAOA depth (circuit layers)
    "optimization_level": 3,  # transpilation effort
    "num_transpiler_runs": 30,  # explore multiple layouts
    "classical_optimizer": "COBYLA",  # robust default for this landscape
    "classical_optimizer_options": {
        "maxiter": 60  # practical convergence budget
    },
    # You can pass backend-specific options; leaving None uses least-busy routing
    "estimator_options": None,
    "sampler_options": None,
}

print("Configured hardware optimization profile:")
for key, value in optimizer_options.items():
    print(f"  {key}: {value}")

Configured hardware optimization profile:
  simulator: False
  num_solutions: 100000
  reps: 3
  optimization_level: 3
  num_transpiler_runs: 30
  classical_optimizer: COBYLA
  classical_optimizer_options: {'maxiter': 60}
  estimator_options: None
  sampler_options: None


## Step 3: Execute using Qiskit primitives

We now execute the full workflow using the Singularity function’s `create_fit_predict` action to train, optimize, and evaluate the `QuantumEnhancedEnsembleClassifier` end-to-end on IBM infrastructure. The function builds the ensemble, applies quantum optimization through Qiskit primitives, and returns both predictions and job metadata (including runtime and resource usage). The classical data split from Step 1 is reused for reproducibility, with validation data passed through `fit_params` so the optimization can tune hyperparameters internally while keeping the held-out test set untouched.

In this step, we explore several configurations of the quantum ensemble to understand how key parameters - specifically `num_learners` and `regularization` - affect both result quality and QPU usage.
- `num_learners` determines the ensemble width (and implicitly, the number of qubits), influencing the model’s capacity and computational cost.
- `regularization` controls sparsity and overfitting, shaping how many learners remain active after optimization.

By varying these parameters, we can see how ensemble width and regularization interact: increasing width typically improves F1 but costs more QPU time, while stronger or adaptive regularization can improve generalization at roughly the same hardware footprint. The next subsections walk through three representative configurations to illustrate these effects.

### Baseline

This configuration uses `num_learners = 10` and `regularization = 7`.

- `num_learners` controls the ensemble width — effectively the number of weak learners combined and, on quantum hardware, the **number of qubits required**. A larger value expands the combinatorial search space and can improve accuracy and recall, but also increases circuit width, compilation time, and overall QPU usage.
- `regularization` sets the penalty strength for including additional learners. With the default "onsite" regularization, higher values enforce stronger sparsity (fewer learners kept), while lower values allow more complex ensembles.

This setup provides a low-cost baseline, showing how a small ensemble behaves before scaling width or tuning sparsity.

In [9]:
# Problem scale and regularization
NUM_LEARNERS = 10
REGULARIZATION = 7

In [10]:
# ----- Quantum-enhanced ensemble on IBM hardware -----
print("\n-- Submitting quantum-enhanced ensemble job --")
job_1 = singularity.run(
    action="create_fit_predict",
    name="grid_stability_qeec",
    quantum_classifier="QuantumEnhancedEnsembleClassifier",
    num_learners=NUM_LEARNERS,
    regularization=REGULARIZATION,
    optimizer_options=optimizer_options,  # from Step 2
    backend_name=backend,  # least-busy compatible backend
    instance=IBM_INSTANCE_QUANTUM,
    random_state=RANDOM_STATE,
    X_train=X_train_bal,
    y_train=y_train_bal,
    X_test=X_test,
    fit_params={"validation_data": (X_val, y_val)},
    options={"save": False},
)
result_1 = job_1.result()
print("Action status:", result_1.get("status"))
print("Action message:", result_1.get("message"))
print("Metadata:", result_1.get("metadata"))
qeec_pred_job_1 = np.array(result_1["data"]["predictions"])
_ = evaluate_predictions(qeec_pred_job_1, y_test)


-- Submitting quantum-enhanced ensemble job --
Action status: ok
Action message: Classifier created, fitted, and predicted.
Metadata: {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 267.05158376693726}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 3336.8785166740417}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 152.4274561405182}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 1550.1889700889587}}}
Accuracy: 0.868
Precision: 1.0
Recall: 0.868
F1: 0.9293361884368309


In [11]:
status_1 = job_1.status()
print("\nQuantum job status:", status_1)


Quantum job status: DONE


## Step 2: 양자 하드웨어 실행을 위한 문제 최적화
앙상블 선택 작업은 각 약한 학습기가 이진 결정 변수인 조합 최적화 문제로 구성되며, 목적 함수는 정규화 항을 통해 정확도와 희소성의 균형을 맞춥니다. `QuantumEnhancedEnsembleClassifier`는 IBM 하드웨어에서 QAOA로 이 문제를 해결하면서도 시뮬레이터 기반 탐색을 허용합니다. `optimizer_options`는 하이브리드 루프를 제어합니다: `simulator=False`는 Circuit를 선택한 QPU로 라우팅하고, `num_solutions`는 탐색 범위를 넓히며, `classical_optimizer_options`(내부 고전 최적화기용)는 수렴을 제어합니다; 약 60회 반복 값이 품질과 런타임의 균형에 좋습니다. 런타임 옵션(적당한 Circuit 깊이(`reps`) 및 표준 트랜스파일 노력 등)은 장치 전반에 걸쳐 강건한 성능을 보장하는 데 도움이 됩니다. 아래 구성은 하드웨어 실행에 사용할 "최적 결과" 프로파일입니다; `simulator=True`로 전환하여 QPU 시간을 소비하지 않고 워크플로를 사전 테스트하는 순수 시뮬레이션 변형을 만들 수도 있습니다.

In [12]:
# Problem scale and regularization
NUM_LEARNERS = 30
REGULARIZATION = 7

In [13]:
# ----- Quantum-enhanced ensemble on IBM hardware -----
print("\n-- Submitting quantum-enhanced ensemble job --")
job_2 = singularity.run(
    action="create_fit_predict",
    name="grid_stability_qeec",
    quantum_classifier="QuantumEnhancedEnsembleClassifier",
    num_learners=NUM_LEARNERS,
    regularization=REGULARIZATION,
    optimizer_options=optimizer_options,  # from Step 2
    backend_name=backend,  # least-busy compatible backend
    instance=IBM_INSTANCE_QUANTUM,
    random_state=RANDOM_STATE,
    X_train=X_train_bal,
    y_train=y_train_bal,
    X_test=X_test,
    fit_params={"validation_data": (X_val, y_val)},
    options={"save": False},
)
result_2 = job_2.result()
print("Action status:", result_2.get("status"))
print("Action message:", result_2.get("message"))
print("QPU Time:", result_2.get("metadata"))
qeec_pred_job_2 = np.array(result_2["data"]["predictions"])
_ = evaluate_predictions(qeec_pred_job_2, y_test)


-- Submitting quantum-enhanced ensemble job --
Action status: ok
Action message: Classifier created, fitted, and predicted.
QPU Time: {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 680.2116754055023}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 80.80395102500916}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 154.4466371536255}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 1095.822762966156}}}
Accuracy: 0.8946666666666667
Precision: 1.0
Recall: 0.8946666666666667
F1: 0.944405348346235


In [14]:
status_2 = job_2.status()
print("\nQuantum job status:", status_2)


Quantum job status: DONE


### Regularization

In this configuration, we increase to `num_learners = 60` and introduce adaptive regularization to manage sparsity more intuitively.

- With `regularization = "auto"`, the optimizer automatically finds a suitable regularization strength that selects approximately `regularization_ratio * num_learners` weak learners for the final ensemble, rather than fixing the penalty manually. This provides a more convenient interface for managing the balance between sparsity and ensemble size.
- `regularization_type = "alpha"` defines how the penalty is applied. Unlike `onsite`, which is unbounded `[0, ∞]`, `alpha` is bounded between `[0, 1]`, making it easier to tune and interpret. The parameter controls the trade-off between individual and pairwise penalties, offering a smoother configuration range.
- `regularization_desired_ratio ≈ 0.82` specifies the target proportion of learners to keep active after regularization — here, around 82% of learners are retained, trimming the weakest 18% automatically.

While adaptive regularization simplifies configuration and helps maintain a balanced ensemble, it does not necessarily guarantee better or more stable performance. The actual quality depends on selecting an appropriate regularization parameter, and fine-tuning it through cross-validation can be computationally expensive. The main advantage lies in improved usability and interpretability rather than direct accuracy gains.

In [15]:
# Problem scale and regularization
NUM_LEARNERS = 60
REGULARIZATION = "auto"
REGULARIZATION_TYPE = "alpha"
REGULARIZATION_RATIO = 0.82

In [16]:
# ----- Quantum-enhanced ensemble on IBM hardware -----
print("\n-- Submitting quantum-enhanced ensemble job --")
job_3 = singularity.run(
    action="create_fit_predict",
    name="grid_stability_qeec",
    quantum_classifier="QuantumEnhancedEnsembleClassifier",
    num_learners=NUM_LEARNERS,
    regularization=REGULARIZATION,
    regularization_type=REGULARIZATION_TYPE,
    regularization_desired_ratio=REGULARIZATION_RATIO,
    optimizer_options=optimizer_options,  # from Step 2
    backend_name=backend,  # least-busy compatible backend
    instance=IBM_INSTANCE_QUANTUM,
    random_state=RANDOM_STATE,
    X_train=X_train_bal,
    y_train=y_train_bal,
    X_test=X_test,
    fit_params={"validation_data": (X_val, y_val)},
    options={"save": False},
)
result_3 = job_3.result()
print("Action status:", result_3.get("status"))
print("Action message:", result_3.get("message"))
print("Metadata:", result_3.get("metadata"))
qeec_pred_job_3 = np.array(result_3["data"]["predictions"])
_ = evaluate_predictions(qeec_pred_job_3, y_test)


-- Submitting quantum-enhanced ensemble job --
Action status: ok
Action message: Classifier created, fitted, and predicted.
Metadata: {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 1387.7451872825623}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 95.41597843170166}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 171.78878355026245}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 1146.5584812164307}}}
Accuracy: 0.908
Precision: 1.0
Recall: 0.908
F1: 0.9517819706498952


In [17]:
status_3 = job_3.status()
print("\nQuantum job status:", status_3)


Quantum job status: DONE


## Step 4: Post-process and return result in desired classical format

We now post-process outputs from both the classical and quantum runs, converting them into a consistent format for downstream evaluation. This step compares predictive quality using standard metrics - accuracy, precision, recall, and F1 - and analyzes how ensemble width (`num_learners`) and sparsity control (`regularization`) influence both performance and computational behavior.

The classical AdaBoost baseline provides a compact and stable reference for small-scale learning. It performs well with limited ensembles and negligible compute overhead, reflecting the strength of traditional boosting when the hypothesis space is still tractable. The quantum configurations (`qeec_pred_job_1`, `qeec_pred_job_2`, and `qeec_pred_job_3`) extend this baseline by embedding the ensemble-selection process within a variational quantum optimization loop. This allows the system to explore exponentially large subsets of learners simultaneously in superposition, addressing the combinatorial nature of ensemble selection more efficiently as scale increases.

Results show that increasing `num_learners` from 10 to 30 improves recall and F1, confirming that a wider ensemble captures richer interactions among weak learners. The gain is sublinear on current hardware - each additional learner yields smaller accuracy increments - but the underlying scaling behavior remains favorable because the quantum optimizer can search broader configuration spaces without the exponential blow-up typical of classical subset selection. Regularization introduces additional nuance: a fixed λ=7 enforces consistent sparsity and stabilizes convergence, whereas adaptive α-regularization automatically tunes sparsity based on correlations between learners. This dynamic pruning often achieves slightly higher F1 for the same qubit width, balancing model complexity and generalization.

When compared directly with the AdaBoost baseline, the smallest quantum configuration (L=10) reproduces similar accuracy, validating the hybrid pipeline’s correctness. At larger widths, quantum variants - especially with auto-regularization - begin to surpass the classical baseline modestly, showing improved recall and F1 without linear growth in computational cost. These improvements do not indicate immediate "quantum advantage" but rather **scaling efficiency**: the quantum optimizer maintains tractable performance as the ensemble expands, where a classical approach would face exponential growth in subset-selection complexity.

In practice:
- Use the **classical baseline** for quick validation and benchmarking on small datasets.
- Apply **quantum ensembles** when model width or feature complexity grows—QAOA-based search scales more gracefully in those regimes.
- Employ **adaptive α-regularization** to maintain sparsity and generalization without increasing circuit width.
- Monitor QPU time and depth to balance quality gains against near-term hardware constraints.

Together, these experiments show that quantum-optimized ensembles complement classical methods: they reproduce baseline accuracy at small scales while offering a path to efficient scaling on larger, combinatorial learning problems. As hardware improves, these scaling advantages are expected to compound, extending the feasible size and depth of ensemble-based models beyond what is classically practical.

### Evaluate metrics for each configuration

We now evaluate all configurations - the classical AdaBoost baseline and the three quantum ensembles - using the `evaluate_predictions` helper to compute accuracy, precision, recall, and F1 on the same test set. This comparison clarifies how quantum optimization scales relative to the classical approach: at small widths, both perform similarly; as ensembles grow, the quantum method can explore larger hypothesis spaces more efficiently. The resulting table captures these trends in a consistent, quantitative form.

In [18]:
results = []

# Classical baseline
acc_b, prec_b, rec_b, f1_b = evaluate_predictions(baseline_pred, y_test)
results.append(
    {
        "Config": "AdaBoost (Classical)",
        "Accuracy": acc_b,
        "Precision": prec_b,
        "Recall": rec_b,
        "F1": f1_b,
    }
)

# Quantum runs
for label, preds in [
    ("QEEC L=10, reg=7", qeec_pred_job_1),
    ("QEEC L=30, reg=7", qeec_pred_job_2),
    (f"QEEC L=60, reg=auto (α={REGULARIZATION_RATIO})", qeec_pred_job_3),
]:
    acc, prec, rec, f1 = evaluate_predictions(preds, y_test)
    results.append(
        {
            "Config": label,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
        }
    )


df_results = pd.DataFrame(results)
df_results

Accuracy: 0.7893333333333333
Precision: 1.0
Recall: 0.7893333333333333
F1: 0.8822652757078987
Accuracy: 0.868
Precision: 1.0
Recall: 0.868
F1: 0.9293361884368309
Accuracy: 0.8946666666666667
Precision: 1.0
Recall: 0.8946666666666667
F1: 0.944405348346235
Accuracy: 0.908
Precision: 1.0
Recall: 0.908
F1: 0.9517819706498952


,Config,Accuracy,Precision,Recall,F1
0,AdaBoost (Classical),0.789333,1.0,0.789333,0.882265
1,"QEEC L=10, reg=7",0.868000,1.0,0.868000,0.929336
2,"QEEC L=30, reg=7",0.894667,1.0,0.894667,0.944405
3,"QEEC L=60, reg=auto (α=0.82)",0.908000,1.0,0.908000,0.951782


### 학습기 수 증가
여기서는 `regularization = 7`을 유지하면서 `num_learners`를 10 → 30으로 늘립니다.

- 더 많은 학습기는 가설 공간을 확장하여 모델이 더 미묘한 패턴을 포착할 수 있게 하며, F1을 소폭 향상시킬 수 있습니다.
- 대부분의 경우 학습기 10개와 30개 사이의 런타임 차이는 크지 않으며, 이는 Circuit 너비 증가가 실행 비용을 크게 늘리지 않음을 나타냅니다.
- 품질 향상은 여전히 *수확 체감 곡선*을 따릅니다. 앙상블이 성장할수록 초기 이득이 나타나지만, 추가 학습기가 새로운 정보를 덜 제공함에 따라 평탄해집니다.

이 실험은 품질-효율성 트레이드오프를 부각시킵니다. Backend 및 트랜스파일 조건에 따라 앙상블 너비를 늘리면 주요 런타임 패널티 없이 작은 정확도 이득을 제공할 수 있습니다.

In [19]:
x = np.arange(len(df_results))
width = 0.35
plt.figure(figsize=(7.6, 4.6))
plt.bar(x - width / 2, df_results["Accuracy"], width=width, label="Accuracy")
plt.bar(x + width / 2, df_results["F1"], width=width, label="F1")
plt.xticks(x, df_results["Config"], rotation=10)
plt.ylabel("Score")
plt.title("Classical vs Quantum ensemble performance")
plt.legend()
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sml-classification/extracted-outputs/0f15c5fb-2450-4671-9bc2-471043414df2-0.avif" alt="Output of the previous code cell" />

### Interpretation

The plot confirms the expected scaling pattern. The classical AdaBoost performs strongly for smaller ensembles but becomes increasingly costly to scale as the number of weak learners grows, because its subset-selection problem expands combinatorially. The quantum-enhanced models replicate classical accuracy at low widths and begin to surpass it as ensemble size increases, especially under adaptive α-regularization. This reflects the quantum optimizer’s ability to sample and evaluate many candidate subsets in parallel through superposition, maintaining tractable search even at higher widths. While current hardware overhead offsets some of the theoretical gains, the trend illustrates the scaling efficiency advantage of the quantum formulation. In practical terms, the classical method remains preferable for lightweight benchmarks, while quantum-enhanced ensembles become advantageous as model dimensionality and ensemble size expand, offering better trade-offs between accuracy, generalization, and computational growth.

## Appendix: Scaling benefits and enhancements

The scalability advantage of the `QuantumEnhancedEnsembleClassifier` arises from how the ensemble-selection process maps to quantum optimization.
Classical ensemble learning methods, such as AdaBoost or random forests, become computationally expensive as the number of weak learners increases because selecting the optimal subset is a combinatorial problem that scales exponentially.

In contrast, the quantum formulation — implemented here via the Quantum Approximate Optimization Algorithm (QAOA) — can explore these exponentially large search spaces more efficiently by evaluating multiple configurations in superposition.
As a result, the training time does not grow significantly with the number of learners, allowing the model to remain efficient even as ensemble width increases.

While current hardware introduces some noise and depth limitations, this workflow demonstrates a near-term hybrid approach where classical and quantum components cooperate: the quantum optimizer provides a better initialization landscape for the classical loop, improving convergence and final model quality.
As quantum processors evolve, these scalability benefits are expected to extend to larger datasets, broader ensembles, and deeper circuit depths.

## References

1. [Introduction to Qiskit Functions](/docs/guides/functions)
2. [Multiverse Computing Singularity Machine Learning](/docs/guides/multiverse-computing-singularity)

## Tutorial survey

Please take a minute to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_3BLFkNVEuh0QBWm)